# VNC Desktop Demo

Starts a virtual desktop inside the container and opens it in your browser via noVNC.

```
Xvfb (virtual display) → x11vnc (VNC server) → websockify (WebSocket bridge) → browser via noVNC
```
Run the cells in order.

In [ ]:
# Cell 1: Start the VNC stack
import subprocess, os, time
from IPython.display import HTML, display

DISPLAY_NUM = 20
VNC_PORT    = 5920
NOVNC_PORT  = 6920

# Clean up any previous run
os.system(f"pkill -f 'Xvfb :{DISPLAY_NUM}'; "
          f"pkill -f 'x11vnc.*{VNC_PORT}'; "
          f"pkill -f 'websockify.*{NOVNC_PORT}'")
time.sleep(1)

env = {**os.environ, "DISPLAY": f":{DISPLAY_NUM}"}

subprocess.Popen(["Xvfb", f":{DISPLAY_NUM}", "-screen", "0", "1280x1024x24", "-ac"])
time.sleep(1)

subprocess.Popen([
    "x11vnc", "-display", f":{DISPLAY_NUM}",
    "-rfbport", str(VNC_PORT), "-forever", "-nopw", "-quiet"
])
time.sleep(1)

subprocess.Popen([
    "websockify", "--web", "/usr/share/novnc/",
    str(NOVNC_PORT), f"localhost:{VNC_PORT}"
])
time.sleep(1)

# noVNC defaults its WebSocket path to just "/websockify".
# Through jupyter-server-proxy we must pass the full path explicitly.
base = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/")
ws_path = f"{base.lstrip("/")}proxy/{NOVNC_PORT}/websockify"
url = f"{base}proxy/{NOVNC_PORT}/vnc.html?autoconnect=true&resize=scale&path={ws_path}"

print(f"VNC stack started")
display(HTML(f'<a href="{url}" target="_blank" style="font-size:1.2em">&#128444; Open VNC Desktop</a>'))
print(f"Direct URL: {url}")

## Launch a graphical application

Run either cell — the window appears in the VNC desktop tab.

In [ ]:
# xeyes — classic X11 demo
import subprocess, os
subprocess.Popen(["xeyes"], env={**os.environ, "DISPLAY": ":20"})
print("xeyes launched — switch to the VNC desktop tab")

In [ ]:
# matplotlib window via Tk backend
import subprocess, os
script = """
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
import numpy as np
fig, ax = plt.subplots(figsize=(7, 5))
theta = np.linspace(0, 2*np.pi, 500)
for r in [1, 2, 3]:
    ax.plot(r*np.cos(theta), r*np.sin(theta), label=f"track r={r}")
ax.set_aspect("equal")
ax.set_title("Toy event display (matplotlib/Tk)")
ax.legend()
plt.show()
"""
subprocess.Popen(["python3", "-c", script], env={**os.environ, "DISPLAY": ":20"})
print("matplotlib window launched — switch to the VNC desktop tab")

In [ ]:
# Cleanup
import os
os.system("pkill xeyes; pkill x11vnc; pkill websockify; pkill Xvfb")
print("VNC stack stopped")